In [4]:
import pandas as pd
import numpy as np
import glob, os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [1]:
# Jupyter (notebook is inside notebooks/):
RAW = "../data/raw"

# Colab: uncomment these 2 lines instead and adjust the folder path
# from google.colab import drive; drive.mount('/content/drive')
# RAW = "/content/drive/MyDrive/supply-chain-analysis/data/raw"

In [5]:
files = sorted(glob.glob(os.path.join(RAW, "*.csv")))
tables = {}

for f in files:
    name = os.path.splitext(os.path.basename(f))[0]
    tables[name] = pd.read_csv(f)

print(f"Loaded {len(tables)} tables:\n")
for name in tables:
    print(name)

Loaded 12 tables:

carriers.csv
fulfillment_status.csv
purchase_order_lines.csv
purchase_orders.csv
raw_materials.csv
sales_orders.csv
shipment_tracking.csv
stock_movements.csv
supplier_performance.csv
suppliers.csv
warehouse_inventory.csv
warehouses.csv


In [6]:
overview = pd.DataFrame({
    "table": list(tables.keys()),
    "rows": [df.shape[0] for df in tables.values()],
    "columns": [df.shape[1] for df in tables.values()],
    "missing_cells": [int(df.isna().sum().sum()) for df in tables.values()],
    "duplicate_rows": [int(df.duplicated().sum()) for df in tables.values()],
})
overview

,table,rows,columns,missing_cells,duplicate_rows
0,carriers.csv,4,2,0,0
1,fulfillment_status.csv,5000,4,777,161
2,purchase_order_lines.csv,3141,5,527,116
3,purchase_orders.csv,1250,6,297,40
4,raw_materials.csv,4,4,1,0
5,sales_orders.csv,5000,6,1003,143
6,shipment_tracking.csv,5000,6,1182,136
7,stock_movements.csv,712,6,170,18
8,supplier_performance.csv,4,5,1,0
9,suppliers.csv,4,3,0,1


In [7]:
def inspect(name):
    df = tables[name]
    print(f"===== {name} =====")
    print("Shape:", df.shape, "\n")
    display(df.head())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values per column:")
    print(df.isna().sum()[df.isna().sum() > 0])

# Run once per table, replacing with your actual names from Cell 3:
inspect(list(tables.keys())[0])

===== carriers.csv =====
Shape: (4, 2) 



,CarrierID,CarrierName
0,CAR-01,ChocoExpress
1,CAR-02,SweetHaul Logistics
2,CAR-03,Global Confections Freight
3,CAR-04,RapidRoute



Data types:
CarrierID      object
CarrierName    object
dtype: object

Missing values per column:
Series([], dtype: int64)


In [8]:
for name, df in tables.items():
    print(f"\n===== {name} ({df.shape[0]} rows) =====")

    # Date columns stored as text
    for col in df.columns:
        if "date" in col.lower() and df[col].dtype == "object":
            print(f"[DATE AS TEXT] {col}  e.g. {df[col].dropna().iloc[:2].tolist()}")

    # Text columns with messy variants (case / spaces)
    for col in df.select_dtypes(include="object").columns:
        if "date" in col.lower():
            continue
        n_raw = df[col].nunique()
        n_clean = df[col].dropna().astype(str).str.strip().str.lower().nunique()
        if n_raw != n_clean:
            print(f"[INCONSISTENT TEXT] {col}: {n_raw} variants -> {n_clean} after cleanup")
        if n_raw <= 10:
            print(f"[VALUES] {col}: {df[col].value_counts(dropna=False).to_dict()}")

    # Negative numbers
    for col in df.select_dtypes(include="number").columns:
        neg = (df[col] < 0).sum()
        if neg:
            print(f"[NEGATIVE] {col}: {neg} rows")

    # Duplicate IDs
    first = df.columns[0]
    if df[first].duplicated().sum():
        print(f"[DUPLICATE ID] {first}: {df[first].duplicated().sum()}")


===== carriers.csv (4 rows) =====
[VALUES] CarrierID: {'CAR-01': 1, 'CAR-02': 1, 'CAR-03': 1, 'CAR-04': 1}
[VALUES] CarrierName: {'ChocoExpress': 1, 'SweetHaul Logistics': 1, 'Global Confections Freight': 1, 'RapidRoute': 1}

===== fulfillment_status.csv (5000 rows) =====
[INCONSISTENT TEXT] FulfillmentID: 4596 variants -> 4585 after cleanup
[INCONSISTENT TEXT] SalesOrderID: 4614 variants -> 4606 after cleanup
[INCONSISTENT TEXT] ShipmentID: 4608 variants -> 4601 after cleanup
[INCONSISTENT TEXT] Status: 2 variants -> 1 after cleanup
[VALUES] Status: {'Fulfilled': 4560, ' Fulfilled ': 240, nan: 200}
[DUPLICATE ID] FulfillmentID: 403

===== purchase_order_lines.csv (3141 rows) =====
[INCONSISTENT TEXT] POLineID: 2872 variants -> 2868 after cleanup
[INCONSISTENT TEXT] PO_ID: 1330 variants -> 1222 after cleanup
[INCONSISTENT TEXT] MaterialID: 8 variants -> 4 after cleanup
[VALUES] MaterialID: {'RM02': 772, 'RM04': 715, 'RM03': 697, 'RM01': 689, nan: 110, ' RM04 ': 45, ' RM02 ': 42, ' RM0